In [ ]:
# Установка необходимых пакетов
!pip install sentence-transformers requests numpy


In [ ]:
#!/usr/bin/env python3
"""
Нейро-сотрудник для подбора автомобильных запчастей
"""

import sqlite3
import numpy as np
import requests
from sentence_transformers import SentenceTransformer
from typing import List, Dict

# БАЗА ЗНАНИЙ
KNOWLEDGE_BASE = """
## Тормозная система
### Тормозные колодки
**Фрагмент**: Тормозные колодки - ключевой элемент безопасности. Подбор зависит от марки, модели, года выпуска и типа тормозной системы (передние/задние, дисковые/барабанные). Важно учитывать материал изготовления - органические, полуметаллические, керамические. Замена рекомендуется при толщине фрикционного слоя менее 2-3 мм.

### Тормозные диски
**Фрагмент**: Тормозные диски подвержены износу и требуют замены при достижении минимальной толщины. Важно учитывать диаметр и тип вентиляции (сплошные, вентилируемые). Рекомендуется замена дисков парами на одной оси. Минимальная толщина указывается производителем на торце диска.

### Тормозная жидкость
**Фрагмент**: Тормозная жидкость гигроскопична и требует замены каждые 2 года или 40-60 тыс. км. Используются жидкости стандартов DOT4, DOT5.1. При подборе важно учитывать рекомендации производителя автомобиля.

## Фильтры
### Воздушные фильтры
**Фрагмент**: Воздушные фильтры защищают двигатель от пыли и загрязнений. Замена рекомендуется каждые 15-30 тыс. км в зависимости от условий эксплуатации. При подборе учитывают размеры корпуса и тип фильтрующего элемента (бумажные, хлопковые).

### Масляные фильтры
**Фрагмент**: Масляные фильтры обеспечивают чистоту моторного масла. При замене важно использовать фильтры с правильной резьбой и уплотнительным кольцом. Замена осуществляется вместе с моторным маслом каждые 10-15 тыс. км.

### Салонный фильтр
**Фрагмент**: Салонный фильтр очищает воздух, поступающий в салон автомобиля. Различают простые противопылевые и угольные фильтры. Замена рекомендуется каждые 15-20 тыс. км или раз в год.

## Подвеска и рулевое управление
### Амортизаторы
**Фрагмент**: Амортизаторы влияют на комфорт и устойчивость автомобиля. При подборе учитывают тип (гидравлические, газовые) и способ крепления. Рекомендуется замена парами на одной оси.

### Шаровые опоры
**Фрагмент**: Шаровые опоры обеспечивают подвижное соединение в подвеске. Критически важны для безопасности - требуют регулярной диагностики. Люфт в шаровой опоре недопустим.

### Стойки стабилизатора
**Фрагмент**: Стойки стабилизатора уменьшают крен кузова в поворотах. При износе появляется стук при проезде неровностей. Замена рекомендуется при обнаружении люфта.

## Двигатель и выхлопная система
### Свечи зажигания
**Фрагмент**: Свечи зажигания обеспечивают воспламенение топливно-воздушной смеси. При подборе учитывают калильное число, размер резьбы и тип электродов. Замена рекомендуется согласно регламенту ТО (обычно 30-60 тыс. км).

### Ремень ГРМ
**Фрагмент**: Ремень ГРМ синхронизирует работу коленвала и распредвалов. Обрыв ремня приводит к серьезным повреждениям двигателя. Замена строго по регламенту производителя (обычно 60-120 тыс. км или 5-7 лет).

### Катализатор
**Фрагмент**: Катализатор снижает токсичность выхлопных газов. При выходе из строя увеличивается расход топлива и ухудшается динамика. При подборе учитывают совместимость с системой управления двигателем.

## Электрика и освещение
### Аккумулятор
**Фрагмент**: Аккумулятор обеспечивает запуск двигателя и питание бортовой сети. При подборе учитывают емкость (Ач), пусковой ток (А) и полярность. Рекомендуется замена при падении напряжения ниже 12В под нагрузкой.

### Лампы головного света
**Фрагмент**: Лампы головного света обеспечивают освещение дороги. Различают галогенные, ксеноновые и светодиодные лампы. При замене важно соблюдать установочные размеры и цоколь.

### Предохранители
**Фрагмент**: Предохранители защищают электрические цепи от перегрузки. Номинал предохранителя указывается на корпусе. Замена на предохранитель большего номинала недопустима.
"""

class AutoPartsConsultant:
    def __init__(self, deepseek_api_key: str):
        self.deepseek_api_key = deepseek_api_key
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.db_path = "auto_parts.db"
        self.embedding_size = 384
        self.is_running = True
        self.initialize_database()
        self.load_knowledge_from_string(KNOWLEDGE_BASE)

    def initialize_database(self):
        """Инициализация векторной базы данных"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS knowledge_chunks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                level1_title TEXT NOT NULL,
                level2_title TEXT NOT NULL,
                content TEXT NOT NULL,
                embedding BLOB
            )
        ''')

        conn.commit()
        conn.close()
        print("✅ База данных инициализирована")

    def load_knowledge_from_string(self, knowledge_text: str):
        """Загрузка знаний из строки"""
        chunks = self.parse_knowledge_structure(knowledge_text)

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('DELETE FROM knowledge_chunks')

        for chunk in chunks:
            text_for_embedding = f"{chunk['level1']} {chunk['level2']} {chunk['content']}"
            embedding = self.model.encode(text_for_embedding)

            cursor.execute('''
                INSERT INTO knowledge_chunks (level1_title, level2_title, content, embedding)
                VALUES (?, ?, ?, ?)
            ''', (
                chunk['level1'],
                chunk['level2'],
                chunk['content'],
                embedding.tobytes()
            ))

        conn.commit()
        conn.close()
        print(f"✅ Загружено {len(chunks)} чанков знаний")

    def parse_knowledge_structure(self, content: str) -> List[Dict]:
        """Парсинг структуры знаний с двумя уровнями"""
        chunks = []
        lines = content.split('\n')

        current_level1 = ""
        current_level2 = ""
        current_content = []

        for line in lines:
            line = line.strip()
            if not line:
                continue

            if line.startswith('## '):
                if current_content and current_level1 and current_level2:
                    chunks.append({
                        'level1': current_level1,
                        'level2': current_level2,
                        'content': ' '.join(current_content)
                    })
                current_level1 = line[3:].strip()
                current_level2 = ""
                current_content = []

            elif line.startswith('### '):
                if current_content and current_level1 and current_level2:
                    chunks.append({
                        'level1': current_level1,
                        'level2': current_level2,
                        'content': ' '.join(current_content)
                    })
                current_level2 = line[4:].strip()
                current_content = []

            elif line.startswith('**Фрагмент**:'):
                if current_content and current_level1 and current_level2:
                    chunks.append({
                        'level1': current_level1,
                        'level2': current_level2,
                        'content': ' '.join(current_content)
                    })
                current_content = [line[12:].strip()]
            elif current_content:
                current_content.append(line)

        if current_content and current_level1 and current_level2:
            chunks.append({
                'level1': current_level1,
                'level2': current_level2,
                'content': ' '.join(current_content)
            })

        return chunks

    def search_knowledge(self, query: str, top_k: int = 3) -> List[Dict]:
        """Поиск релевантных чанков знаний"""
        query_embedding = self.model.encode(query)

        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        cursor.execute('SELECT level1_title, level2_title, content, embedding FROM knowledge_chunks')
        results = cursor.fetchall()
        conn.close()

        similarities = []
        for row in results:
            stored_embedding = np.frombuffer(row[3], dtype=np.float32)

            if len(stored_embedding) != len(query_embedding):
                min_len = min(len(stored_embedding), len(query_embedding))
                stored_embedding = stored_embedding[:min_len]
                query_embedding_adj = query_embedding[:min_len]
            else:
                query_embedding_adj = query_embedding

            similarity = np.dot(query_embedding_adj, stored_embedding) / (
                np.linalg.norm(query_embedding_adj) * np.linalg.norm(stored_embedding)
            )
            similarities.append({
                'level1': row[0],
                'level2': row[1],
                'content': row[2],
                'similarity': similarity
            })

        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        return similarities[:top_k]

    def call_deepseek_api(self, messages: List[Dict]) -> str:
        """Вызов DEEP SEEK API"""
        if not self.is_running:
            return "Сервис остановлен"

        url = "https://api.deepseek.com/v1/chat/completions"

        headers = {
            "Authorization": f"Bearer {self.deepseek_api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": "deepseek-chat",
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 2000
        }

        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            result = response.json()
            return result['choices'][0]['message']['content']
        except Exception as e:
            return f"Ошибка при обращении к API: {str(e)}"

    def process_customer_request(self, customer_input: str) -> str:
        """Обработка запроса клиента"""
        if not self.is_running:
            return "Сервис остановлен"

        relevant_knowledge = self.search_knowledge(customer_input, top_k=3)

        context = "Релевантная информация из базы знаний:\n\n"
        for knowledge in relevant_knowledge:
            context += f"{knowledge['level1']} - {knowledge['level2']}\n"
            context += f"{knowledge['content']}\n\n"

        system_prompt = f"""
Ты - консультант по подбору автомобильных запчастей. Используй информацию из базы знаний для точных консультаций.

Информация из базы знаний:
{context}

Инструкции:
1. Используй информацию из базы знаний для ответов
2. Уточняй марку, модель, год и VIN-номер автомобиля при необходимости
3. Предлагай варианты запчастей с техническим обоснованием
4. Будь точным и профессиональным
5. Если информации недостаточно - задавай уточняющие вопросы

Отвечай как профессиональный консультант.
"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": customer_input}
        ]

        return self.call_deepseek_api(messages)

    def stop_service(self):
        """Остановка сервиса"""
        self.is_running = False
        print(" Сервис остановлен")

def main():

    api_key = "sk-581d1b95cdc54cccb88799db66c0e821"

    # Инициализация консультанта
    consultant = AutoPartsConsultant(api_key)

    print("=" * 70)
    print("🚗 НЕЙРО-СОТРУДНИК ДЛЯ ПОДБОРА АВТОЗАПЧАСТЕЙ")
    print("=" * 70)
    print(" Консультант готов к работе")
    print("")
    print(" БЫСТРАЯ ОСТАНОВКА: Введите 'СТОП' в любой момент")
    print(" Другие команды остановки: 'выход', 'stop', 'quit'")
    print("")
    print(" Примеры вопросов:")
    print("   • Нужны тормозные колодки для Toyota Camry 2018")
    print("   • Подберите воздушный фильтр для BMW X5")
    print("   • Какие свечи зажигания для Ford Focus 2015?")
    print("=" * 70)

    try:
        while consultant.is_running:
            user_input = input("\n👤 Клиент: ").strip()

            # Проверяем команды остановки
            stop_commands = ['стоп', 'выход', 'exit', 'quit', 'stop', 'stop', 'закончить']
            if user_input.lower() in stop_commands:
                consultant.stop_service()
                print("👋 До свидания!")
                break

            if user_input:
                print(" Думаю...")
                response = consultant.process_customer_request(user_input)
                print(f"\n Консультант: {response}")

    except KeyboardInterrupt:
        print("\n\n Остановлено по Ctrl+C")
        consultant.stop_service()

if __name__ == "__main__":
    main()

✅ База данных инициализирована
✅ Загружено 15 чанков знаний
🚗 НЕЙРО-СОТРУДНИК ДЛЯ ПОДБОРА АВТОЗАПЧАСТЕЙ
 Консультант готов к работе

 БЫСТРАЯ ОСТАНОВКА: Введите 'СТОП' в любой момент
 Другие команды остановки: 'выход', 'stop', 'quit'

 Примеры вопросов:
   • Нужны тормозные колодки для Toyota Camry 2018
   • Подберите воздушный фильтр для BMW X5
   • Какие свечи зажигания для Ford Focus 2015?

👤 Клиент: нужен предохранитель на фары Toyota LiteAce 
 Думаю...

 Консультант: Здравствуйте! Конечно, помогу вам с подбором предохранителя на фары для Toyota LiteAce.

На основании предоставленной информации я могу дать вам общее направление для поиска. Однако для точного подбора мне потребуются дополнительные данные, так как предохранители различаются по номиналу (силе тока) и типу в зависимости от конкретной модели и года выпуска автомобиля.

**Что нам известно и что важно учитывать:**

*   Как указано в базе знаний: **номинал предохранителя указывается на его корпусе**. Замена на предохраните

Описание проделанной работы и цель проекта

В данной домашней работе была реализована интерактивная нейросистема-консультант по подбору автомобильных запчастей.
Цель проекта — создать умного ассистента, который может:

понимать естественные запросы клиентов;

искать релевантную информацию в собственной базе знаний;

использовать API языковой модели DeepSeek для генерации осмысленных, профессиональных ответов;

помогать при подборе деталей по описанию.

Что было сделано пошагово

Создана база знаний
Знания описаны в структуре Markdown с разделами: тормозная система, фильтры, подвеска, двигатель, электрика и т. д.
Для каждой категории добавлены текстовые описания — «фрагменты», из которых формируются смысловые чанки.

Построена векторная база
Использована модель SentenceTransformer('all-MiniLM-L6-v2') для кодирования текстов в эмбеддинги.
Эти векторы сохраняются в SQLite-базе, что позволяет находить наиболее релевантные записи по смысловому сходству.

Реализован поиск по смыслу (semantic search)
При вводе запроса клиента формируется вектор запроса, затем вычисляется косинусное сходство между ним и сохранёнными эмбеддингами.
Три наиболее релевантных фрагмента из базы знаний передаются модели DeepSeek.

Интеграция с DeepSeek API
Используется метод chat/completions.
Ассистент получает контекст из базы знаний и формулирует ответ в стиле профессионального консультанта, предлагая варианты деталей и уточняя недостающие данные (год, модель, VIN и т. д.).

Интерактивный консольный интерфейс
Пользователь может задавать вопросы в режиме диалога.